# Tool Calling and Structured Execution

Large Language Models (LLMs) are text-prediction engines. They cannot inherently query a database, check real-time stock, or perform accurate math. 

To give LLMs these capabilities, we use **Tool Calling** (or Function Calling). We provide the model with a list of our local Python functions. When the user asks a question, the LLM does not run the code; it generates a structured JSON object telling *us* which function to run and what arguments to use.

Tool calling (also called function calling) is the mechanism that allows an LLM to request structured actions from the surrounding system. The model decides *what* to call and *with which arguments*; the system decides *how* to execute it.

This preserves model flexibility, system safety, deterministic execution.

### Conceptual Execution Loop


1. **User intent** is expressed in natural language
2. **Model reasoning** determines whether a tool is required
3. **Structured tool request** is generated (name + arguments)
4. Local system executes the tool
5. Result is optionally fed back to the model


This loop is the foundation for agentic systems, and retrieval + action pipelines

In [12]:
import os
import json
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Load API credentials securely
load_dotenv(override=True)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY2")

# Initialize the Gemini Client
client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL_ID = "gemini-3.5-flash-lite"

## Step 1: Defining the Local Tool

A tool is an ordinary Python function with a strict contract. For the LLM to understand how to use this function, we must include strict type hints (e.g., `product_name: str`) and a clear Docstring. The API uses these elements to automatically generate the JSON schema for the tool.

For Gemini to convert a Python function into a callable tool, it relies on:
* Type hints
* Docstring description
* Function signature

These are used to automatically derive:
* Input schema
* Argument types
* Tool semantics

The model never sees your function body. It only sees the schema derived from it. First, let's define a standard Python function. 

In [13]:
def check_inventory(product_name: str) -> str:
    """
    Checks the database for product availability.
    """
    database = {
        "laptop": "28 units in stock",
        "headphones": "Out of stock"
    }

    print(f"function called for {product_name}")        # this will help us log when the function is called

    # Simple dictionary lookup
    return database.get(product_name.lower(), "Product not found.")

# Test it locally first
print("Local Python test:", check_inventory("laptop"))

function called for laptop
Local Python test: 28 units in stock


Before exposing a function to an LLM, we need to ensure that it is deterministic to avoid confusing model errors with function bugs.

## Step 2: The LLM Generates the Tool Call

At this stage, we don't want free-form text. We want the model to pause generation and request a tool. This is achieved by supplying tools in the configuration to constrain the behaviour using a system instruction.


### Passing the Callable to LLM

Next, we pass our `check_inventory` function to the LLM via the configuration parameters. We will ask a question that requires checking the stock.

Notice that we are going to inspect the **raw response object**, because if the LLM recognises the need of a tool, it will *suspend* text generation and return a `function_calls` object instead.

In [14]:
user_query = "Please check the database to see if we have headphones in stock."

response = client.models.generate_content(
    model=MODEL_ID,
    contents=user_query,
    config=types.GenerateContentConfig(
        tools=[check_inventory],
        system_instruction = "You are an inventory assistant. You must use the provided tools to check stock levels. Do not guess.",
        temperature=0.0     # Setting the temperature to 0 to ensure reliability, a good practice
    )
)

print("Generated Text:", bool(response.text))
print("Requested Tool:", bool(response.function_calls))

if response.function_calls:
    print("Raw Tool Output:", response.function_calls[0].name)

function called for headphones
Generated Text: True
Requested Tool: False


In [15]:
response.text

'Headphones are currently out of stock.'

The LLM was able to answer the question correctly. Here, we see that the output says the model did not call the tool, but if you see the log message we added, it shows that the function was actually called.

This is because the Google GenAI SDK (Software Development Kit) handles **automatic function calling** by default, executing the tool and feeding the results back to the model in a hidden background loop before delivering the final text to you. Essentially, the model sees your function's output, but the SDK is the one that actually looks inside the function's body to run the code.


We can also disable this automatic behaviour.

In [16]:
# Let's try using a random item, elephant, and laptop
user_query = "Please check the database to see if we have laptop or elephant in stock."

response = client.models.generate_content(
    model=MODEL_ID,
    contents=user_query,
    config=types.GenerateContentConfig(
        tools=[check_inventory],
        automatic_function_calling={"disable": True},
        system_instruction = "You are an inventory assistant. You must use the provided tools to check stock levels. Do not guess.",
        temperature=0.0    
    )
)

print("Generated Text:", response.text) 
print("Requested Tool:", bool(response.function_calls))

if response.function_calls:
    fc = response.function_calls[0]
    print(f"Model wants to call: {fc.name} with args: {fc.args}")

Generated Text: None
Requested Tool: True
Model wants to call: check_inventory with args: {'product_name': 'laptop'}


Now, we can see that the tool is actually called. But now the model does not respond with text. It only gives the name of the tool to be called. You will soon see how this is sent back in the loop to get the final response from the model.

*Note: If the user query contains "laptops" instead of "laptop", the model may extract "laptops" as the argument value. If the database key is strictly "laptop" and no normalisation is applied, this can cause the lookup to fail and result in a response such as “There are no laptops in stock”. This behaviour depends on the model, the tool schema, and how argument extraction is defined.*


<br>

For a model to call a tool reliably, at least one of these must be true:
1. The model cannot see the data
2. The model cannot compute the result
3. The model is forbidden from answering without a tool call


To ensure reliable tool usage, we also need to provide a detailed description of the function and its arguments to the model. It helps the model to create a suitable schema for the tool.

### Using a Function Declaration

Another method of using tools is through explicit declaration and tool registration. For this, we first declare the function using a schema and them define the function declarations as tools. You can refer to the [function calling documentation here](https://ai.google.dev/gemini-api/docs/function-calling?example=meeting#how-it-works).

In [17]:
# Creating a function declaration for the 'check_inventory' function
# It contains the keys "name", "description", "parameters"
inventory_fn = types.FunctionDeclaration(
    name="check_inventory",
    description="Checks an inventory system for product availability.",
    parameters={
        "type": "object",
        "properties": {
            "product_name": {
                "type": "string",
                "description": "Name of the product to check"
            }
        },
        "required": ["product_name"]
    }
)

# Passing all the function decalarations (only one, here) as a list
inventory_tool = types.Tool(function_declarations=[inventory_fn])

In [18]:
user_query = "Please check the database to see if we have laptop or elephant in stock."

response = client.models.generate_content(
    model=MODEL_ID,
    contents=user_query,
    config=types.GenerateContentConfig(
        tools=[inventory_tool],     # Pass the declared tools
        system_instruction = "You are an inventory assistant. You must use the provided tools to check stock levels. Do not guess.",
        temperature=0.0  
    )
)

print("Generated Text:", bool(response.text))
print("Requested Tool:", bool(response.function_calls))

if response.function_calls:
    print("Raw Tool Output:", response.function_calls[0].name)

Generated Text: False
Requested Tool: True
Raw Tool Output: check_inventory


## Step 3: Extracting the Arguments

Let us look closely at the `function_calls` object the LLM returned. It contains a dictionary `args` showing the values to pass for each argument of the function, extracted from natural language query, and the `name` of the function to call.

In [19]:
response.function_calls

[FunctionCall(
   args={
     'product_name': 'laptop'
   },
   id='call_2502238',
   name='check_inventory'
 ),
 FunctionCall(
   args={
     'product_name': 'elephant'
   },
   id='call_2502239',
   name='check_inventory'
 )]

In [20]:
if response.function_calls:
    for tool_request in response.function_calls:
    # we are defining one instance of function call as one tool request here

        requested_name = tool_request.name
        extracted_args = tool_request.args

        print("Function requested :", requested_name)
        print("Arguments extracted:", extracted_args)
else:
    print("No function was requested. Model generated text instead.")

Function requested : check_inventory
Arguments extracted: {'product_name': 'laptop'}
Function requested : check_inventory
Arguments extracted: {'product_name': 'elephant'}


As you saw, a tool request contains:
* Tool name → routing
* Arguments dictionary → structured extraction

This is where natural language to structured data conversion occurs. The model will identify parameter names and map text spans to values.

## Step 4: Local Execution

The LLM is currently waiting. It has asked us to run `check_inventory` with the arguments `{'product_name': 'laptop'}` and `{'product_name': 'elephant'}`.

We will now execute our Python function using these extracted arguments and store the result.

In [21]:
local_results = []

if response.function_calls:
    for tool_request in response.function_calls:
    
        requested_name = tool_request.name  # doing this again as we didn't store it anywhere for simplicity
        extracted_args = tool_request.args

        # We unpack the dictionary using **extracted_args
        local_result = check_inventory(**extracted_args)

        local_results.append(local_result)

        print(f"Executing locally: {requested_name}({extracted_args})")
        print("Result computed by Python:", local_result)

local_results   # the final list of both function call results

function called for laptop
Executing locally: check_inventory({'product_name': 'laptop'})
Result computed by Python: 28 units in stock
function called for elephant
Executing locally: check_inventory({'product_name': 'elephant'})
Result computed by Python: Product not found.


['28 units in stock', 'Product not found.']

At this point, the model is idle, while the system is responsible for execution. This boundary prevents arbitrary code execution and avoids any model-initiated side effects.

## Step 5: Providing Tool Results to the LLM

We asked the model to use the provided tools. Therefore, it stops after emitting tool calls. 

The model cannot see the tool results unless you explicitly send them back.

So, we send the original user query, as well as the response from each tool. Each function call is paired with its name.

In [22]:
tool_messages = []

for tool_request, tool_result in zip(response.function_calls, local_results):

    # Each tool message has a conversation role (like "user", "assistant"), and message parts
    # Message parts are specific Part objects which contain the function responses: details of tool request name and a `response` dict storing the tool result
    
    msg_parts = types.Part(function_response=types.FunctionResponse(name=tool_request.name, 
                                                                    response={"result": tool_result}))

    message = types.Content(
                            role="tool",
                            parts=[msg_parts])

    tool_messages.append(message)

tool_messages

[Content(
   parts=[
     Part(
       function_response=FunctionResponse(
         name='check_inventory',
         response={
           'result': '28 units in stock'
         }
       )
     ),
   ],
   role='tool'
 ),
 Content(
   parts=[
     Part(
       function_response=FunctionResponse(
         name='check_inventory',
         response={
           'result': 'Product not found.'
         }
       )
     ),
   ],
   role='tool'
 )]

We must also explicitly include the model’s earlier tool requests to give the model sufficient context

In [42]:
# Similarly, role and parts for the tool requests (we only need the function call details here)
function_call_content = types.Content(role="model",
                                            parts=[types.Part(function_call=types.FunctionCall(name=tool_request.name,
                                                                                                args=tool_request.args)) for tool_request in response.function_calls])

Now we can send the tool messages as the usual contents

In [43]:
final_response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
            types.Content(role="user", parts=[types.Part(text=user_query)]),    # we also need to convert the user query to the Part style object now
            function_call_content, *tool_messages],
    config=types.GenerateContentConfig(
        temperature=0.0)
)

print(final_response.text)

We have 28 laptops in stock. Unfortunately, we do not have any elephants in stock.


So, to demonstrate the entire pipeline:
```
User
↓
Model (requests tool)
↓
Tool (returns result)
↓
Model (produces final text)
```

## Semantic Routing

Manually extracting arguments, running the code, and sending the result back to the LLM requires writing a lot of boilerplate code. 

In production, we use the SDK's built-in automatic function calling. By using a `chats` session, the SDK will automatically handle the back-and-forth loop between the LLM and our local Python functions.

When multiple tools are available, the model performs **semantic routing**. It selects the tool whose description best matches the intent, and no explicit rules or if-else logic is required. This is a key capability that distinguishes tool calling from classical API pipelines.


In [44]:
# Let's add a second tool, a calculator
def calculate_discount(price: float) -> float:
    """Applies a 20 percent staff discount to a price."""

    print(f"discount calculated for {price}")   # logging the function call
    
    return price * 0.80

We can use the `client.chats.create` directly instead of using dictionaries with `client.models.generate_content` to directly create a chat-style session. 

It is still just an abstraction of `client.models.generate_content`, but makes the multi-turn conversations easy to handle. You can simply use the `chat.send_message()` method to add user-side messages to the chat. Refer to the [documentation here](https://ai.google.dev/gemini-api/docs/text-generation#multi-turn-conversations).

In [45]:
# Initialise an automated chat session with both tools
chat_session = client.chats.create(
    model=MODEL_ID,
    config=types.GenerateContentConfig(
        tools=[check_inventory, calculate_discount],    # directly pass callables
        automatic_function_calling={"disable": False},  # keep function calling automatic
        temperature=0.0
    )
)


reply_1 = chat_session.send_message("I want to buy a laptop. Are they in stock?")
print("Model Response:", reply_1.text)


reply_2 = chat_session.send_message("The laptop is $1000, but I have a staff discount. What is my final price?")
print("Model Response:", reply_2.text)

Model Response: What is the name of the laptop you are looking for?
discount calculated for 1000
Model Response: The final price is $800.


## Tool-Calling - HuggingFace and OpenAI

Additionally, let's see small examples of how we can use tools in OpenAI GPT models and how it can be simulated in huggingface.

In [47]:
from huggingface_hub import InferenceClient
from openai import OpenAI

HF_TOKEN = os.getenv("HF_TOKEN")
hf_client = InferenceClient(token=HF_TOKEN)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
openai = OpenAI(api_key=OPENAI_API_KEY)

#### OpenAI

We need to define a schema here as well. The schema specifies:
* What capability exists (name, description)
* What information is required (parameters)

In [66]:
tools = [
    {
        "type": "function",
        "name": "check_inventory",
        "description": "Checks the database for product availability.",
        "parameters": {
            "type": "object",
            "properties": {
                "product_name": {
                    "type": "string",
                    "description": "Name of the product"
                }
            },
            "required": ["product_name"]
        }
    }
]

In [67]:
response = openai.responses.create(
    model="gpt-4.1-nano",
    input="Do we have headphones in stock?",
    tools=tools,
    tool_choice="auto"
)

Let's see how the model selected a tool, whether it was able to extract the argument and the final result

In [70]:
# Inspect model decision
for item in response.output:
    if item.type == "function_call":
        tool_name = item.name
        args = json.loads(item.arguments)

        print("Tool requested:", tool_name)
        print("Arguments:", args)

        result = check_inventory(**args)
        print("Tool executed locally →", result)

Tool requested: check_inventory
Arguments: {'product_name': 'headphones'}
Tool executed locally → Out of stock


Now, we can again pass this result to the LLM again to get the final responses.

Other than functions, there are other types of tools that can be used: `code_interpreter`, `file_search`, `computer_use`, `image_generation`, `shell` (commands). For example:


In [75]:
response = openai.responses.create(
  model="gpt-4o-mini",
  input="How old will Donald Trump be in July 2026?",
  tools=[{"type": "web_search"}] # Enables the built-in web search tool
)

print(response.output_text)

Donald Trump was born on June 14, 1946. In July 2026, he will turn 80 years old.


#### HuggingFace

Hugging Face inference clients do not execute tools automatically. Tool-calling behaviour is achieved by prompting the model to emit a structured tool request.

In [ ]:
# HF tool calling

tools_description = """
You have access to the following tool:

Tool name: check_inventory
Description: Checks the database for product availability.
Arguments:
- product_name (string)

If a tool is required, respond ONLY in valid JSON like this:
{
  "tool": "check_inventory",
  "arguments": {
    "product_name": "laptop"
  }
}

Otherwise, respond normally.
"""

In [55]:
user_query = "Do we have headphones in stock?"

response = hf_client.chat_completion(
    model="meta-llama/Meta-Llama-3-8B-Instruct",
    messages=[
        {"role": "system", "content": tools_description},
        {"role": "user", "content": user_query}
    ],
    max_tokens=200,
    temperature=0.0
)

assistant_output = response.choices[0].message.content
print("Raw model output:\n", assistant_output)

Raw model output:
 {
  "tool": "check_inventory",
  "arguments": {
    "product_name": "headphones"
  }
}


In [56]:
# System-side handling 
try:
    parsed = json.loads(assistant_output)
    if parsed.get("tool") == "check_inventory":
        result = check_inventory(**parsed["arguments"])
        print("Tool executed locally →", result)
except json.JSONDecodeError:
    print("No tool call detected.")

function called for headphones
Tool executed locally → Out of stock
